# News Clustering (Basic)

Notebook ini memperkenalkan clustering berita menggunakan data Excel Anda.

## Sumber data
- File: 08-Text Mining/news_clf.xlsx
- Sheet: training

## Target belajar
1. Membaca data teks dari Excel
2. Membersihkan teks sederhana
3. Mengubah teks menjadi angka (TF-IDF)
4. Melakukan clustering dengan KMeans
5. Membaca hasil cluster

## 1) Install dan import library

In [ ]:
# Uncomment jika library belum ada
# !pip install pandas openpyxl scikit-learn matplotlib seaborn

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import TruncatedSVD

## 2) Baca data dari Excel (sheet training)

In [ ]:
path_file = "news_clf.xlsx"
sheet_name = "training"

try:
    df = pd.read_excel(path_file, sheet_name=sheet_name)
except ValueError:
    xls = pd.ExcelFile(path_file)
    print("Sheet tidak ditemukan. Daftar sheet tersedia:", xls.sheet_names)
    raise

print("Ukuran data:", df.shape)
df.head()

## 3) Pilih kolom Judul dan Konten
Cell ini mencari kolom judul + konten otomatis. Keduanya akan digabung ke kolom baru bernama MetaData.

In [ ]:
kandidat_judul = ["judul", "title", "headline", "subject"]
kandidat_konten = ["konten", "content", "isi", "news", "text", "berita", "body"]
lower_map = {c.lower(): c for c in df.columns}

title_col = next((lower_map[k] for k in kandidat_judul if k in lower_map), None)
content_col = next((lower_map[k] for k in kandidat_konten if k in lower_map), None)

if title_col is None or content_col is None:
    print("Kolom judul/konten belum terdeteksi otomatis.")
    print("Daftar kolom:", df.columns.tolist())
    print('Contoh manual: title_col = "Judul" ; content_col = "Konten"')
else:
    print("Kolom terpilih -> judul:", title_col, "| konten:", content_col)

## 3b) EDA ringkas sebelum cleaning
Kita cek kondisi data: nilai kosong, duplikasi, dan distribusi panjang teks berita.

In [ ]:
print("=== Info Data Awal ===")
print("Jumlah baris:", len(df))
print("Jumlah duplikasi baris:", df.duplicated().sum())
print("Missing value per kolom:")
print(df.isna().sum())

if title_col is None or content_col is None:
    print("\nEDA panjang teks dilewati karena kolom judul/konten belum terdeteksi.")
else:
    eda_df = df.dropna(subset=[title_col, content_col]).copy()
    eda_df["MetaData"] = (
        eda_df[title_col].astype(str).str.strip() + " " + eda_df[content_col].astype(str).str.strip()
    ).str.replace(r"\s+", " ", regex=True).str.strip()
    eda_df["text_len"] = eda_df["MetaData"].str.len()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    sns.histplot(eda_df["text_len"], bins=20, kde=True, ax=axes[0], color="#457b9d")
    axes[0].set_title("Distribusi Panjang MetaData (Judul + Konten)")
    axes[0].set_xlabel("Jumlah Karakter")
    axes[0].set_ylabel("Frekuensi")

    sns.boxplot(x=eda_df["text_len"], ax=axes[1], color="#a8dadc")
    axes[1].set_title("Ringkasan Panjang MetaData (Boxplot)")
    axes[1].set_xlabel("Jumlah Karakter")

    plt.tight_layout()
    plt.show()

    print("Ringkasan statistik panjang MetaData:")
    display(eda_df["text_len"].describe().to_frame("text_len"))

## 4) Buat kolom MetaData dan cleaning teks sederhana
MetaData = gabungan Judul + Konten, lalu dibersihkan (lowercase, hapus angka/simbol, rapikan spasi).

In [ ]:
def clean_text(s):
    s = str(s).lower()
    s = re.sub(r"[^a-zA-Z\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

if title_col is None or content_col is None:
    raise ValueError("Kolom judul/konten belum tersedia. Set manual title_col dan content_col.")

df_work = df.copy()
df_work = df_work.dropna(subset=[title_col, content_col])
df_work["MetaData"] = (
    df_work[title_col].astype(str).str.strip() + " " + df_work[content_col].astype(str).str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()
df_work["clean_text"] = df_work["MetaData"].apply(clean_text)

df_work[[title_col, content_col, "MetaData", "clean_text"]].head()

## 5) Ubah MetaData ke TF-IDF
TF-IDF mengubah teks pada kolom MetaData (setelah cleaning) menjadi fitur numerik untuk clustering.

In [ ]:
vectorizer = TfidfVectorizer(max_features=1000, stop_words="english")
X = vectorizer.fit_transform(df_work["clean_text"])

print("Sumber TF-IDF: kolom MetaData (setelah cleaning)")
print("Shape TF-IDF:", X.shape)

feature_names = vectorizer.get_feature_names_out()
print("Jumlah kolom fitur TF-IDF:", len(feature_names))
print("Contoh nama kolom fitur:", feature_names[:20].tolist())

# Tampilkan MetaData dan hasil vectorizer (preview agar tetap ringan)
preview_rows = 5
preview_features = 15

meta_preview = df_work[["MetaData"]].head(preview_rows).reset_index(drop=True)
tfidf_preview = pd.DataFrame(
    X[:preview_rows].toarray(),
    columns=feature_names
).iloc[:, :preview_features].round(3)

display(pd.concat([meta_preview, tfidf_preview], axis=1))

## 6) Coba beberapa jumlah cluster (k)
Kita pakai silhouette score untuk gambaran kualitas cluster.

In [ ]:
hasil_k = []
k_values = [2, 3, 4, 5, 6, 7, 8]

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    score = silhouette_score(X, labels)
    hasil_k.append((k, score))

df_sil = pd.DataFrame(hasil_k, columns=["k", "silhouette_score"])
display(df_sil)

plt.figure(figsize=(7, 4))
sns.lineplot(data=df_sil, x="k", y="silhouette_score", marker="o", color="#e76f51")
plt.title("Silhouette Score untuk Berbagai Nilai k")
plt.xlabel("Jumlah Cluster (k)")
plt.ylabel("Silhouette Score")
plt.xticks(k_values)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

k_terbaik = df_sil.loc[df_sil["silhouette_score"].idxmax(), "k"]
print(f"Saran awal dari silhouette: k = {k_terbaik}")

## 6b) Elbow Method (Inertia)
Selain silhouette score, kita lihat penurunan inertia. Titik siku (elbow) biasanya jadi kandidat jumlah cluster yang baik.

In [ ]:
inertia_values = []

for k in k_values:
    km_elbow = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_elbow.fit(X)
    inertia_values.append(km_elbow.inertia_)

df_elbow = pd.DataFrame({
    "k": k_values,
    "inertia": inertia_values
})

display(df_elbow)

plt.figure(figsize=(7, 4))
sns.lineplot(data=df_elbow, x="k", y="inertia", marker="o", color="#264653")
plt.title("Elbow Method: Inertia vs Jumlah Cluster")
plt.xlabel("Jumlah Cluster (k)")
plt.ylabel("Inertia")
plt.xticks(k_values)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Interpretasi: pilih k di titik siku saat penurunan inertia mulai melandai.")

## 7) Clustering final
Untuk kelas basic, kita set k = 5. Boleh diubah sesuai hasil evaluasi silhouette dan elbow method di atas.

In [ ]:
k_final = 3
model = KMeans(n_clusters=k_final, random_state=42, n_init=10)
df_work["cluster"] = model.fit_predict(X)

print(df_work["cluster"].value_counts().sort_index())
df_work[["MetaData", "cluster"]].head(10)

## 7b) Visualisasi hasil cluster
Kita tampilkan jumlah anggota tiap cluster dan proyeksi dokumen ke ruang 2D agar pola cluster lebih mudah dilihat.

In [ ]:
cluster_count = df_work["cluster"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.barplot(x=cluster_count.index.astype(str), y=cluster_count.values, ax=axes[0], palette="Set2")
axes[0].set_title("Distribusi Jumlah Dokumen per Cluster")
axes[0].set_xlabel("Cluster")
axes[0].set_ylabel("Jumlah Dokumen")

svd = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd.fit_transform(X)
plot_df = pd.DataFrame({
    "dim1": X_2d[:, 0],
    "dim2": X_2d[:, 1],
    "cluster": df_work["cluster"].astype(str)
})

sns.scatterplot(data=plot_df, x="dim1", y="dim2", hue="cluster", palette="Set1", ax=axes[1], s=60)
axes[1].set_title("Peta Dokumen 2D berdasarkan Cluster")
axes[1].set_xlabel("Dimensi 1 (SVD)")
axes[1].set_ylabel("Dimensi 2 (SVD)")
axes[1].legend(title="Cluster")

plt.tight_layout()
plt.show()

## 8) Lihat kata kunci utama tiap cluster
Kata dengan bobot tertinggi membantu interpretasi tema cluster.

In [ ]:
terms = vectorizer.get_feature_names_out()
centers = model.cluster_centers_

for i in range(k_final):
    top_idx = centers[i].argsort()[-10:][::-1]
    top_words = [terms[j] for j in top_idx]
    print(f"Cluster {i}:", ", ".join(top_words))

## 9) Simpan hasil clustering

In [ ]:
output_file = "news_clustering_result.csv"
df_work.to_csv(output_file, index=False)
print("Hasil tersimpan di:", output_file)